# COMP5329 — Deep Learning

**Tutorial 7 — Sequence Models: From RNN to Transformer**

**Semester 1, 2026**

### Learning Objectives

By the end of this tutorial you will be able to:

1. Explain why **sequential data** requires specialised architectures (not MLPs or CNNs).
2. Implement a **vanilla RNN** from scratch and understand its hidden-state recurrence.
3. Derive why RNNs suffer from **vanishing/exploding gradients** over long sequences.
4. Implement **LSTM** from scratch and explain how gating mechanisms preserve long-range gradients.
5. Implement **GRU** from scratch and compare its gate structure to LSTM.
6. Compare RNN, LSTM, and GRU on the **same task** with identical training pipelines.
7. Explain the **self-attention** mechanism and why it replaces recurrence.
8. Implement **scaled dot-product attention** and **multi-head attention** from scratch.
9. Build a **Transformer encoder block** (attention + FFN + LayerNorm + residual connections).
10. Train a Transformer on a language modelling task and visualise attention weights.

### Topic Coverage

Week 7 covers **sequence models from RNN to Transformer**. The full topic list (see `Week7_Self_Study.ipynb`) is:

- ✅ **Why sequence models?** — motivation and architectural evolution *(tutorial)*
- ✅ **Data pipeline** — letter-counting benchmark *(tutorial, briefly)*
- 📖 **Vanilla RNN** — hidden-state recurrence, from-scratch implementation *(self-study; live tutorial only discusses the vanishing gradient argument)*
- 📖 **LSTM & GRU** — gating mechanisms, cell-state conveyor belt *(self-study; live tutorial touches on §3.1 theory only)*
- ✅ **The limits of recurrence** — why we need attention *(tutorial)*
- ✅ **Self-attention & scaled dot-product attention** — Q/K/V mechanism, √dₖ scaling *(tutorial, with in-class practice)*
- 📖 **Multi-head attention, positional encoding, encoder block, masking** — full Transformer plumbing *(self-study — code is provided in this notebook as post-tutorial reading)*
- 📖 **Language modelling with `nn.TransformerEncoder`** — end-to-end Penn Treebank run *(self-study extension)*

Due to time constraints, the live tutorial focuses on **self-attention as the key idea that replaces recurrence**, and the in-class practice implements **scaled dot-product attention** from scratch. The recurrent architectures (RNN/LSTM/GRU) are left as self-study — the self-study notebook covers them in depth along with applications (VQA, seq2seq, NMT).

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Walkthrough

## 0. Why Sequence Models?

In previous weeks we studied CNNs (which assume **spatial locality** with fixed receptive fields) and GNNs (which assume **graph-structured** relationships). However, many real-world tasks -- natural language, time series, music, DNA -- involve data with **ordered, variable-length** dependencies. We need architectures that can process sequences while maintaining a summary of past context.

This tutorial walks through the evolution of sequence modelling: from the simplest recurrent unit to the Transformer. Each successive architecture solves a specific limitation of its predecessor.

| Architecture | Input processing | Memory mechanism | Parallelisable? |
|---|---|---|---|
| MLP | All at once (fixed size) | None | Yes |
| CNN | Local windows | Receptive field | Yes |
| **RNN** | One token at a time | Hidden state | No |
| **LSTM** | One token at a time | Gated cell state | No |
| **GRU** | One token at a time | Gated hidden state | No |
| **Transformer** | All at once | Attention weights | Yes |

We will use a **letter-counting task** as a shared benchmark for RNN, LSTM, and GRU, then move to a **language modelling task** to demonstrate the Transformer's strengths.

---

## 1. Data Pipeline

### 1.1 Task Description

Given a random string of lowercase letters (a-z), uppercase letters (A-Z), and digit noise (0-9), predict the **difference** between the count of lowercase and uppercase letters (offset by 29 to make all labels non-negative).

**Examples:**
- `aAA304` → label = 1 - 2 + 29 = 28
- `bbB234BbB` → label = 3 - 3 + 29 = 29
- `ccccccC` → label = 6 - 1 + 29 = 34

This task requires the model to **count** across variable-length sequences while ignoring noise -- a clean test of sequential memory.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────────
import os
import random
import time
import sys
import gc
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.autograd import Variable

%matplotlib inline

### 1.2 Data Generation

In [ ]:
# ── Data generation ─────────────────────────────────────────────────────────────

def generate_line(input_char):
    """Generate a single training example.
    Args:
        input_char: ASCII code of a lowercase letter (ord('a') to ord('z'))
    Returns:
        (string, label) where label = count_lower - count_upper + 29
    """
    num1 = random.randint(1, 30)   # number of lowercase letters
    num2 = random.randint(1, 30)   # number of uppercase letters
    src = [chr(input_char) for _ in range(num1)]        # lowercase
    target = [chr(input_char - 32) for _ in range(num2)] # uppercase
    src.extend(target)

    noise_num = random.randint(0, 100)
    for _ in range(noise_num):
        src.append(str(random.randint(0, 9)))  # digit noise
    random.shuffle(src)

    return ''.join(src), num1 - num2 + 29


def generate_data(size, filename):
    """Generate dataset and write to file."""
    f = open(filename, "w")
    s = set()
    count = 0
    while count < size:
        c = random.randint(ord('a'), ord('z'))
        src, target = generate_line(c)
        if src in s or src[::-1] in s:
            continue
        count += 1
        if count % 10000 == 0:
            print("Generated %d lines" % count)
        s.add(src)
        f.write('\t'.join([src, str(target)]))
        f.write('\n')
    f.close()

generate_data(80000, "seq.txt")

### 1.3 Hyperparameters and Training Utilities

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────────
hidden_size = 50          # hidden state dimension for all RNN variants
embedding_size = 20       # character embedding dimension
input_length = 160        # max sequence length (zero-padded); covers max ~156 chars
vob_size = 52 + 10 + 1    # 26 lower + 26 upper + 10 digits + 1 padding token
output_size = 60          # number of classes (label range ~ 0-58)
max_gradient_norm = 3     # gradient clipping threshold
init_lr_rate = 0.001      # Adam learning rate

MAX_ITERATIONS = 20000    # total training steps (reduced for tutorial)
VAL_INTERVAL = 1000       # validate every N steps
PRINT_INTERVAL = 100      # print loss every N steps
batch_size = 64           # mini-batch size

In [ ]:
# ── Data loading and preprocessing utilities ─────────────────────────────────

def read_dataset(file_name):
    """Read dataset and split into train (64000) and val (16000)."""
    f = open(file_name)
    ls = []
    for line in f.readlines():
        line = line.strip()
        l = line.split('\t')
        ls.append([l[0], int(l[1])])
    random.shuffle(ls)
    return ls[:64000], ls[64000:]


def create_maps():
    """Map characters to integer IDs.
    a-z -> 1-26, A-Z -> 27-52, 0-9 -> 53-62. Index 0 reserved for padding.
    """
    dic = {}
    counter = 1
    for i in range(ord('a'), ord('z') + 1):
        dic[chr(i)] = counter
        counter += 1
    for i in range(ord('A'), ord('Z') + 1):
        dic[chr(i)] = counter
        counter += 1
    for i in range(ord('0'), ord('9') + 1):
        dic[chr(i)] = counter
        counter += 1
    return dic


def word_embedding(input_seq, vob_size, dtype=torch.float):
    """Embed integer sequences into dense vectors."""
    word_embed = nn.Embedding(vob_size, embedding_size)
    embeddings = word_embed(input_seq.long())
    return embeddings


def create_batch(datas, maps):
    """Convert a batch of (string, label) pairs into padded integer tensors."""
    size = len(datas)
    seqs = np.zeros((size, input_length), dtype=np.int32)
    labels = np.zeros(size, dtype=np.int32)
    for i in range(size):
        labels[i] = datas[i][1]
        seq = datas[i][0]
        l = input_length - len(seq)  # left zero-padding
        for j in range(len(seq)):
            seqs[i][l + j] = maps[seq[j]]
    return seqs, labels


maps = create_maps()

In [ ]:
# ── Reusable training and evaluation pipeline ────────────────────────────────
# All models share the same pipeline; only the model class changes.

results = {}  # accumulates {model_name: {"train_loss": [...], "val_acc": [...]}}


def train_and_log(model, model_name, maps):
    """Train a model and log loss/accuracy curves into `results`.
    Args:
        model:      nn.Module with a .predict(input) method
        model_name: string key for results dict (e.g. 'rnn', 'lstm', 'gru')
        maps:       character-to-integer mapping
    """
    train_data, val_data = read_dataset("seq.txt")
    pointer = 0

    model.train()
    optimizer = optim.Adam(model.parameters(), lr=init_lr_rate)
    model_loss = nn.CrossEntropyLoss()

    train_losses = []
    val_accs = []
    running_loss = 0.0

    for step in range(MAX_ITERATIONS + 1):
        # ── Mini-batch sampling ──
        if pointer + batch_size >= len(train_data):
            random.shuffle(train_data)
            pointer = 0
        datas = train_data[pointer:pointer + batch_size]
        pointer += batch_size

        # ── Forward pass ──
        input_seq, label = create_batch(datas, maps)
        input_seq = torch.from_numpy(input_seq)
        label = torch.from_numpy(label)
        seq_emb = word_embedding(input_seq, vob_size)
        input_seq_emb = seq_emb.permute(1, 0, 2)  # (T, B, embed_dim)

        out = model.predict(input_seq_emb)
        step_loss = model_loss(out, label.long())

        # ── Backward pass ──
        optimizer.zero_grad()
        step_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_gradient_norm)
        optimizer.step()

        running_loss += step_loss.item()

        # ── Logging ──
        if step > 0 and step % PRINT_INTERVAL == 0:
            avg_loss = running_loss / PRINT_INTERVAL
            train_losses.append(avg_loss)
            running_loss = 0.0
            if step % (PRINT_INTERVAL * 10) == 0:
                print(f"[{model_name}] step {step}, loss {avg_loss:.3f}")

        # ── Validation ──
        if step % VAL_INTERVAL == 0:
            model.eval()
            val_rate = val_model(model, val_data, maps)
            val_accs.append(val_rate)
            model.train()
            print(f"[{model_name}] step {step}, val accuracy: {val_rate:.3f}")

    results[model_name] = {"train_loss": train_losses, "val_acc": val_accs}
    print(f"\n[{model_name}] Training complete. Final val accuracy: {val_accs[-1]:.3f}")


def val_model(model, dataset, maps):
    """Evaluate model accuracy on a dataset."""
    start_pointer = 0
    total = 0
    with torch.no_grad():
        while start_pointer < len(dataset):
            end_pointer = min(start_pointer + batch_size, len(dataset))
            datas = dataset[start_pointer:end_pointer]
            start_pointer = end_pointer

            input_seq, label = create_batch(datas, maps)
            input_seq = torch.from_numpy(input_seq)
            label = torch.from_numpy(label)
            seq_emb = word_embedding(input_seq, vob_size)
            input_seq_emb = seq_emb.permute(1, 0, 2)

            answers = model.predict(input_seq_emb)
            answer_ids = np.argmax(answers.detach().numpy(), axis=-1)
            total += np.sum(label.detach().numpy() == answer_ids)
    return 1.0 * total / len(dataset)

> **Tutorial note — self-study section.** RNN is covered here as *motivation* for Transformer. In the live tutorial, we only discuss **§2.4 The Vanishing Gradient Problem** — everything else in §2 is for you to read afterwards. The from-scratch RNN implementation is still runnable; treat it as reference code.

---

## 2. Vanilla RNN

### 2.1 Theory

The simplest approach to sequence modelling: at each time step, combine the current input with a summary of everything seen so far.

![RNN Architecture](./Figs/RNN1.png)

The core equations of a vanilla RNN are:

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h)$$
$$y_t = W_{hy}\, h_t + b_y$$

where:
- $x_t \in \mathbb{R}^{d}$ is the input at time step $t$
- $h_t \in \mathbb{R}^{H}$ is the hidden state
- $W_{xh} \in \mathbb{R}^{H \times d}$, $W_{hh} \in \mathbb{R}^{H \times H}$ are the weight matrices

> **Key insight**: The hidden state $h_t$ is a **lossy compression** of the entire input history $x_1, \ldots, x_t$ into a fixed-size vector of $H$ dimensions. Everything the model knows about the past must fit in this vector.

In practice, we concatenate $[x_t; h_{t-1}]$ and use a single weight matrix $W \in \mathbb{R}^{H \times (d+H)}$, which is mathematically equivalent to separate $W_{xh} x_t + W_{hh} h_{t-1}$.

### 2.2 From-Scratch Implementation

In [ ]:
# ── Vanilla RNN ────────────────────────────────────────────────────────────────

class RNNModel(nn.Module):
    def __init__(self, in_feature, hidden_size, n_class):
        super(RNNModel, self).__init__()
        self.in_feature = in_feature
        self.hidden_size = hidden_size
        self.n_class = n_class
        # Single linear layer for concatenated [x_t; h_{t-1}]
        self.fully_connected = nn.Linear(in_feature + self.hidden_size, self.hidden_size)
        self.pred_layer = nn.Linear(self.hidden_size, self.n_class)
        self.tanh = nn.Tanh()

    def forward(self, input, dtype=torch.float):
        T = input.shape[0]          # sequence length
        batch_size = input.shape[1]  # batch size
        outputs = torch.zeros(size=(T, batch_size, self.hidden_size), dtype=dtype)
        state = torch.zeros(size=(batch_size, self.hidden_size), dtype=dtype)  # h_0

        for t in range(T):
            concat = torch.cat([input[t], state], dim=1)  # (B, d+H)
            state = self.tanh(self.fully_connected(concat))  # h_t = tanh(W [x_t; h_{t-1}] + b)
            outputs[t] = state
        return outputs, state  # all hidden states, final hidden state

    def predict(self, input_state, dtype=torch.float):
        _, last_state = self.forward(input_state)  # use final hidden state
        predict = self.tanh(self.pred_layer(last_state))  # (B, n_class)
        return predict

### 2.3 Training

In [ ]:
rnn_model = RNNModel(embedding_size, hidden_size, output_size)
train_and_log(rnn_model, 'rnn', maps)

### 2.4 The Vanishing Gradient Problem

Why does the vanilla RNN struggle with long sequences? Consider the gradient of the loss at time step $T$ with respect to the hidden state at an earlier time step $k$:

![RNN Gradient Vanishing](./Figs/RNN_Gradient.png)

Mathematically, the gradient of $h_T$ with respect to $h_k$ is a product of Jacobians:

$$\frac{\partial h_T}{\partial h_k} = \prod_{t=k+1}^{T} \frac{\partial h_t}{\partial h_{t-1}} = \prod_{t=k+1}^{T} W_{hh}^\top \cdot \text{diag}\big(\tanh'(z_t)\big)$$

Since $|\tanh'(z)| \leq 1$ everywhere (and typically $\ll 1$ for most inputs), and $\|W_{hh}\|$ is often $< 1$ after training, this product **shrinks exponentially** with $T - k$.

**Consequences:**
- Gradients from the loss at step $T$ barely reach step $k$ when $T - k$ is large.
- The model **cannot learn long-range dependencies** -- it effectively "forgets" distant inputs.
- **Gradient clipping** (used in our training pipeline) addresses **exploding** gradients but does nothing for **vanishing** gradients.

> **Transition**: We need a mechanism that allows gradients to flow across many time steps **without multiplicative decay**. The key idea: an **additive** update path.

> **Tutorial note — self-study section.** In the live tutorial, we only discuss **§3.1 Theory** and the **cell-state gradient** argument below — just enough to see why additive updates fix the vanishing gradient. The full 4-gate walk-through and from-scratch implementation are for self-study.

---

## 3. LSTM (Long Short-Term Memory)

### 3.1 Theory

LSTM introduces a **cell state** $c_t$ that acts as a conveyor belt. Information flows along it with only **additive** modifications, not multiplicative. Three **gates** control what enters, what leaves, and what is forgotten.

![LSTM Architecture](./Figs/LSTM.png)

| Gate | Formula | Role |
|---|---|---|
| **Forget** | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | What fraction of old cell state to **keep** |
| **Input** | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | What fraction of new candidate to **add** |
| **Candidate** | $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$ | Proposed **new information** |
| **Cell update** | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ | **Additive** update (the key!) |
| **Output** | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | What part of cell state to **expose** |
| **Hidden** | $h_t = o_t \odot \tanh(c_t)$ | Output hidden state |

### Why LSTM Solves the Vanishing Gradient Problem

Consider the gradient of the cell state across time:

$$\frac{\partial c_T}{\partial c_k} = \prod_{t=k+1}^{T} f_t$$

If the forget gate learns $f_t \approx 1$, this gradient passes through **unchanged** -- no exponential decay! The network **learns** to "not forget" when long-range memory is needed.

This is fundamentally different from vanilla RNN, where the equivalent product involves $W_{hh}$ and $\tanh'(z_t)$, neither of which can be learned to be exactly 1.

**Why these activation functions?**
- **Sigmoid** ($\sigma$) for gates: outputs in $[0, 1]$ act as **soft switches** -- 0 means "block everything", 1 means "pass everything through".
- **Tanh** for candidate $\tilde{c}_t$: centered around 0, allowing the cell state to both **increase and decrease**. This provides balanced gradient flow.

### 3.2 From-Scratch Implementation

In [ ]:
# ── LSTM ──────────────────────────────────────────────────────────────────────

class LSTMModel(nn.Module):
    def __init__(self, in_feature, hidden_size, n_class):
        super(LSTMModel, self).__init__()
        self.in_feature = in_feature
        self.hidden_size = hidden_size
        self.n_class = n_class
        # Single matrix for all 4 gates: [i, f, g, o] concatenated
        self.fully_connected = nn.Linear(self.in_feature + self.hidden_size,
                                          4 * self.hidden_size)
        self.pred_layer = nn.Linear(self.hidden_size, self.n_class)
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()

    def forward(self, input, dtype=torch.float):
        T = input.shape[0]          # sequence length
        batch_size = input.shape[1]  # batch size
        outputs = torch.zeros(size=(T, batch_size, self.hidden_size), dtype=dtype)
        c, h = torch.unbind(torch.zeros([2, batch_size, self.hidden_size]), dim=0)
        # c: cell state (B, H),  h: hidden state (B, H)

        for t in range(T):
            concat = torch.cat([input[t], h], dim=1)    # (B, d+H)
            gates = self.fully_connected(concat)         # (B, 4H)
            i, f, g, o = gates.chunk(4, dim=1)           # each (B, H)

            i_t = self.sigmoid(i)   # input gate
            f_t = self.sigmoid(f)   # forget gate
            g_t = self.tanh(g)      # candidate cell state
            o_t = self.sigmoid(o)   # output gate

            c = f_t * c + i_t * g_t  # cell state update (ADDITIVE!)
            h = o_t * self.tanh(c)   # hidden state
            outputs[t] = h

        return outputs, h

    def predict(self, input_state, dtype=torch.float):
        _, last_state = self.forward(input_state)
        predict = self.pred_layer(last_state)  # (B, n_class)
        return predict

### 3.3 Training and Comparison with RNN

In [ ]:
lstm_model = LSTMModel(embedding_size, hidden_size, output_size)
train_and_log(lstm_model, 'lstm', maps)

In [ ]:
# ── RNN vs LSTM comparison ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name in ['rnn', 'lstm']:
    if name in results:
        ax1.plot(results[name]['train_loss'], label=name.upper())
        ax2.plot(results[name]['val_acc'], label=name.upper())

ax1.set_xlabel('Step (x{})'.format(PRINT_INTERVAL))
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss: RNN vs LSTM')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Validation checkpoint')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy: RNN vs LSTM')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> **Transition**: LSTM effectively addresses vanishing gradients, but it requires **4 parameter matrices** (for 3 gates + candidate). Can we achieve similar gating benefits with fewer parameters?

> **Tutorial note — not covered in class.** GRU is a parameter-efficient variant of LSTM and is *not discussed in the live tutorial*. Read §4 on your own if you are curious; everything you need for Week 8 can be understood from LSTM alone.

---

## 4. GRU (Gated Recurrent Unit) ⭐ *Extension*

> *This section can be covered quickly in class and can be studied in detail afterwards. GRU is architecturally similar to LSTM with fewer parameters.*

### 4.1 Theory

GRU simplifies LSTM by merging the cell state and hidden state into a single $h_t$, and using **2 gates** instead of 3.

![GRU Architecture](./Figs/GRU.png)

| Gate | Formula | Role |
|---|---|---|
| **Reset** | $r_t = \sigma(W_r [h_{t-1}, x_t] + b_r)$ | How much past to forget when computing candidate |
| **Update** | $z_t = \sigma(W_z [h_{t-1}, x_t] + b_z)$ | Interpolation between old and new (replaces both forget and input gates) |
| **Candidate** | $\tilde{h}_t = \tanh(W_h [r_t \odot h_{t-1}, x_t] + b_h)$ | Proposed new state |
| **State update** | $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$ | **Single gate** controls both forgetting and input |

**Key insight**: The update gate $z_t$ plays the role of **both** the forget gate and the input gate in LSTM. When $z_t \approx 0$, the old state passes through unchanged (analogous to $f_t \approx 1$ in LSTM). This is a more **parameter-efficient** design.

**Parameter count comparison** (recurrent layer, with $H = 50$, $d = 20$):
- LSTM: $4 \times H \times (H + d) = 4 \times 50 \times 70 = 14{,}000$ parameters
- GRU: $3 \times H \times (H + d) = 3 \times 50 \times 70 = 10{,}500$ parameters
- GRU uses **25% fewer** recurrent parameters than LSTM.

### 4.2 From-Scratch Implementation

In [ ]:
# ── GRU ───────────────────────────────────────────────────────────────────────

class GRUModel(nn.Module):
    def __init__(self, in_feature, hidden_size, n_class):
        super(GRUModel, self).__init__()
        self.in_feature = in_feature
        self.hidden_size = hidden_size
        self.n_class = n_class
        # fc_1: computes reset (r) and update (z) gates together
        self.fully_connected_1 = nn.Linear(in_feature + self.hidden_size,
                                            2 * self.hidden_size)
        # fc_2: computes candidate state using reset-gated hidden state
        self.fully_connected_2 = nn.Linear(in_feature + self.hidden_size,
                                            self.hidden_size)
        self.pred_layer = nn.Linear(self.hidden_size, self.n_class)
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()

    def forward(self, input, dtype=torch.float):
        T = input.shape[0]          # sequence length
        batch_size = input.shape[1]  # batch size
        outputs = torch.zeros(size=(T, batch_size, self.hidden_size), dtype=dtype)
        state = torch.zeros(size=(batch_size, self.hidden_size), dtype=dtype)  # h_0

        for t in range(T):
            concat = torch.cat([input[t], state], dim=1)           # (B, d+H)
            gates = self.fully_connected_1(concat)                  # (B, 2H)
            r, z = gates.chunk(2, dim=1)                            # each (B, H)

            r_t = self.sigmoid(r)    # reset gate
            z_t = self.sigmoid(z)    # update gate

            concat_reset = torch.cat([input[t], r_t * state], dim=1)  # (B, d+H)
            h_tilde = self.tanh(self.fully_connected_2(concat_reset))  # candidate

            state = (1 - z_t) * state + z_t * h_tilde  # interpolation
            outputs[t] = state

        return outputs, state

    def predict(self, input_state, dtype=torch.float):
        _, last_state = self.forward(input_state)
        predict = self.pred_layer(last_state)  # (B, n_class)
        return predict

### 4.3 Training and Three-Way Comparison

In [ ]:
gru_model = GRUModel(embedding_size, hidden_size, output_size)
train_and_log(gru_model, 'gru', maps)

In [ ]:
# ── Three-way comparison: RNN vs LSTM vs GRU ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {'rnn': '#e74c3c', 'lstm': '#3498db', 'gru': '#2ecc71'}
for name in ['rnn', 'lstm', 'gru']:
    if name in results:
        ax1.plot(results[name]['train_loss'], label=name.upper(), color=colors[name])
        ax2.plot(results[name]['val_acc'], label=name.upper(), color=colors[name])

ax1.set_xlabel('Step (x{})'.format(PRINT_INTERVAL))
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Validation checkpoint')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Summary table ──
print("\n" + "="*60)
print(f"{'Model':<10} {'Params':<12} {'Final Val Acc':<15}")
print("="*60)
for name, model in [('RNN', rnn_model), ('LSTM', lstm_model), ('GRU', gru_model)]:
    n_params = sum(p.numel() for p in model.parameters())
    final_acc = results[name.lower()]['val_acc'][-1]
    print(f"{name:<10} {n_params:<12,} {final_acc:<15.3f}")
print("="*60)

---

## 5. The Limits of Recurrence

Even with gating, all three models share a fundamental constraint: **sequential processing**. Token $t$ must wait for token $t-1$ to finish. This creates three problems:

1. **No parallelism**: Training time scales linearly with sequence length. GPUs can't process all time steps simultaneously.
2. **Information bottleneck**: The entire source sequence must compress into a fixed-size hidden state $h_T \in \mathbb{R}^H$.
3. **Long-range dependencies are still hard**: Even LSTM struggles when $T > 200$--$500$, because the forget gate must learn to stay close to 1 over hundreds of steps.

What if we could let **every token attend to every other token** simultaneously, without any recurrence?

> This is the core idea behind the **Transformer** (Vaswani et al., "Attention is All You Need", 2017).

---

## 6. Transformer

### 6.1 Self-Attention Mechanism

The self-attention mechanism identifies the relationship between **any two tokens** in the input, regardless of their distance.

**Intuition**: Consider the sentence *"The FBI agent who was responsible is now retired."* The verb *"is"* depends on the subject *"agent"* (not the closer *"responsible"*). Self-attention can capture this directly.

#### Query, Key, and Value

For each input token $x_i$, we compute three vectors via learned linear projections:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

where $X \in \mathbb{R}^{T \times d_{model}}$ and $W_Q, W_K, W_V \in \mathbb{R}^{d_{model} \times d_k}$.

- **Query** ($Q$): "What am I looking for?"
- **Key** ($K$): "What do I contain?"
- **Value** ($V$): "What information do I provide?"

The attention output for each query is a **weighted sum of values**, where the weights come from the similarity between the query and all keys.

#### Scaled Dot-Product Attention

![Self-attention computation](./Figs/Self-attention.png)

The attention mechanism computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Step by step:
1. **Similarity scores**: $S = QK^\top \in \mathbb{R}^{T \times T}$ -- dot product between each query and all keys.
2. **Scaling**: Divide by $\sqrt{d_k}$ to prevent large magnitudes.
3. **Softmax**: Normalise each row to get attention weights $A \in \mathbb{R}^{T \times T}$ where each row sums to 1.
4. **Weighted sum**: $\text{Output} = AV \in \mathbb{R}^{T \times d_k}$ -- each output token is a mixture of all value vectors.

![Self-attention information extraction](./Figs/Self-attention2.png)

#### Why $\frac{1}{\sqrt{d_k}}$?

Without scaling, the dot products $q_i \cdot k_j$ grow proportionally to $d_k$ (each is a sum of $d_k$ terms). Large magnitudes push the softmax into **saturation**, where gradients vanish. Dividing by $\sqrt{d_k}$ keeps the variance of the dot products at approximately 1.

> **Key advantage over RNNs**: The attention matrix $A$ connects **every pair of tokens** directly -- information doesn't need to travel through a chain of hidden states. This eliminates the vanishing gradient problem for long-range dependencies.

---
# Part B · In-Class Exercise

## Scaled Dot-Product Attention

**Your task:** implement `scaled_dot_product_attention(Q, K, V, mask=None)` from scratch using only `torch` primitives (no `nn.MultiheadAttention`, no `F.scaled_dot_product_attention`). You will then verify your implementation against a reference on random inputs and visualise the attention pattern with and without a causal mask.

**Shape convention:**

| Tensor | Shape | Meaning |
|---|---|---|
| `Q`, `K`, `V` | `(B, H, T, d_k)` | batch · heads · tokens · per-head dim |
| `mask` | `(T, T)` or broadcastable | `1` = keep, `0` = block |
| output | `(B, H, T, d_k)` | attended values |
| `attn_weights` | `(B, H, T, T)` | post-softmax weights |

**Learning goals:**
1. Internalise the Q/K/V shape dance and why `K` is transposed on its last two axes.
2. Understand *exactly* where `√d_k` goes and why (see Exam Q1).
3. Understand why masked positions are set to `-∞` *before* softmax, not `0` *after*.

> **Rule:** Try the scaffold below first. Open the "Reference solution" cell *only* after you have a working version or you are truly stuck.

In [ ]:
# ── Exercise: Scaled Dot-Product Attention (fill in the TODOs) ───────────────

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Your from-scratch implementation.
    Args:
        Q, K, V: (B, H, T, d_k) tensors
        mask:    (T, T) or broadcastable tensor, or None.
                 1 = keep, 0 = block.
    Returns:
        output:       (B, H, T, d_k)
        attn_weights: (B, H, T, T), each row sums to 1
    """
    d_k = Q.size(-1)

    # TODO 1: Compute raw attention scores by matmul of Q and K.
    #         Expected shape: (B, H, T, T). Hint: transpose the last two axes of K.
    scores = None  # <-- replace

    # TODO 2: Scale by sqrt(d_k). Ask yourself WHY before you write the line.
    #         (See Exam Q1 — the variance argument.)
    scores = None  # <-- replace

    # TODO 3: If a mask is provided, block the masked positions.
    #         Hint: use masked_fill with float('-inf'). Why -inf and not 0?
    if mask is not None:
        scores = None  # <-- replace

    # TODO 4: Softmax over the correct axis, then weighted sum of V.
    #         Ask yourself: which axis are the "keys" on?
    attn_weights = None  # <-- replace
    output = None  # <-- replace

    return output, attn_weights


In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────
# Run this cell after filling in the TODOs above.

torch.manual_seed(42)

# -- (a) Shape check on a realistic batch --------------------------------------
B, H, T, d_k = 2, 4, 6, 16
Q = torch.randn(B, H, T, d_k)
K = torch.randn(B, H, T, d_k)
V = torch.randn(B, H, T, d_k)

out, w = scaled_dot_product_attention(Q, K, V)
assert out.shape == (B, H, T, d_k), f"output shape wrong: {out.shape}"
assert w.shape == (B, H, T, T),     f"attn shape wrong: {w.shape}"
assert torch.allclose(w.sum(dim=-1), torch.ones(B, H, T), atol=1e-5), "rows must sum to 1"
print("(a) shape + row-sum check   OK")

# -- (b) Correctness vs. an independent reference ------------------------------
def _ref(Q, K, V, mask=None):
    d_k = Q.size(-1)
    s = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        s = s.masked_fill(mask == 0, float("-inf"))
    a = F.softmax(s, dim=-1)
    return a @ V, a

ref_out, ref_w = _ref(Q, K, V)
assert torch.allclose(out, ref_out, atol=1e-6), "output mismatch vs reference"
assert torch.allclose(w,   ref_w,   atol=1e-6), "attn mismatch vs reference"
print("(b) correctness vs reference OK")

# -- (c) Causal mask: future positions must be exactly zero --------------------
causal = torch.tril(torch.ones(T, T))            # 1 on/below diagonal, 0 above
out_m, w_m = scaled_dot_product_attention(Q, K, V, mask=causal)
upper_mass = w_m.triu(diagonal=1).abs().max().item()
assert upper_mass < 1e-6, f"mask leak: max upper-triangular weight = {upper_mass}"
print("(c) causal mask check        OK")

# -- (d) Visualise the attention pattern (batch 0, head 0) ---------------------
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, weights, title in [
    (axes[0], w.detach()[0, 0],   "No mask"),
    (axes[1], w_m.detach()[0, 0], "Causal mask"),
]:
    im = ax.imshow(weights, cmap="viridis", vmin=0, vmax=weights.max().item())
    ax.set_title(title); ax.set_xlabel("key position"); ax.set_ylabel("query position")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


#### Stretch task (for fast finishers) — bridge to Multi-Head Attention

Wrap your `scaled_dot_product_attention` function in a `MultiHeadAttention` module:

1. Take input `x` of shape `(B, T, d_model)`.
2. Project it to `Q`, `K`, `V` with three `nn.Linear(d_model, d_model)` layers.
3. Reshape each to `(B, H, T, d_k)` where `d_k = d_model // H`.
4. Call **your** `scaled_dot_product_attention`.
5. Concatenate heads back to `(B, T, d_model)` and apply an output projection.

A skeleton is provided below. If you finish it, read §6.3 as a self-check — the reference implementation there should match yours line-for-line in spirit.


In [ ]:
# ── Stretch: wrap your function in a multi-head module ──────────────────────

class MyMultiHead(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.H = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, _ = x.shape

        # TODO A: project x to Q, K, V with shapes (B, T, d_model)
        Q = None  # <-- replace
        K = None  # <-- replace
        V = None  # <-- replace

        # TODO B: reshape to (B, H, T, d_k)
        #         Hint: .view(B, T, H, d_k).transpose(1, 2)
        Q = None  # <-- replace
        K = None  # <-- replace
        V = None  # <-- replace

        # TODO C: call your scaled_dot_product_attention
        out, attn = None, None  # <-- replace

        # TODO D: concatenate heads back to (B, T, d_model) and apply W_o
        out = None  # <-- replace
        return out, attn


#### 🔒 Reference solution — *open only after you have tried the exercise*

The cell below contains the reference implementation of `scaled_dot_product_attention`. It is **source-hidden by default** so that you do not accidentally read it before attempting the exercise. Click the cell (or use your notebook's "show cell source" option) to reveal it. After revealing, you can run it to overwrite your own version and continue into §6.3.


In [ ]:
# ── Reference solution — scaled_dot_product_attention ───────────────────────

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Reference implementation. Matches the exercise scaffold above."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1))          # (B, H, T, T)
    scores = scores / math.sqrt(d_k)                        # scale — see Exam Q1
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn_weights = F.softmax(scores, dim=-1)                # (B, H, T, T)
    output = torch.matmul(attn_weights, V)                  # (B, H, T, d_k)
    return output, attn_weights


### 6.3 Multi-Head Attention

A single attention head can only focus on **one type of relationship** at a time. Multi-head attention runs $h$ parallel attention computations with **different learned projections**, then concatenates the results. Each head can specialise: one head may attend to syntactic structure, another to semantic similarity.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$
$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

**Design choice**: $d_k = d_v = d_{model} / h$. Since each head operates on a smaller dimension, the total computation is roughly the same as single-head attention with full $d_{model}$ -- but the model gains **multiple perspectives**.

In [ ]:
# ── Multi-Head Attention ─────────────────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)  # projects to all heads at once
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # output projection

    def forward(self, Q, K, V, mask=None):
        """Args: Q, K, V each (B, T, d_model). Returns: (B, T, d_model), attn_weights."""
        B, T, _ = Q.size()

        # ── Project and reshape to (B, num_heads, T, d_k) ──
        Q = self.W_q(Q).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        # ── Scaled dot-product attention (reuses function from 6.2) ──
        out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # ── Concatenate heads: (B, T, d_model) ──
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out), attn_weights

In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────
torch.manual_seed(42)
d_model, num_heads, T = 64, 4, 6
x = torch.randn(2, T, d_model)  # batch of 2 sequences, 6 tokens each
mha = MultiHeadAttention(d_model, num_heads)
out, attn = mha(x, x, x)  # self-attention: Q=K=V=x
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Attention shape: {attn.shape}  (batch, heads, T, T)")

### 6.4 Positional Encoding

Unlike RNNs, the Transformer processes all tokens **simultaneously** -- it has no notion of sequence order. We inject position information by adding a **positional encoding** to the input embeddings.

$$PE_{pos, 2i} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{pos, 2i+1} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**Why sine and cosine?** For any fixed offset $k$, $PE_{pos+k}$ can be expressed as a **linear function** of $PE_{pos}$. This gives the model a systematic way to compute **relative positions** from absolute position encodings. Different dimensions oscillate at different frequencies, creating a unique "fingerprint" for each position.

In [ ]:
# ── Positional Encoding ──────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)                              # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0) / d_model))          # (d_model/2,)
        pe[:, 0::2] = torch.sin(position * div_term)   # even dimensions: sin
        pe[:, 1::2] = torch.cos(position * div_term)   # odd dimensions: cos
        pe = pe.unsqueeze(0).transpose(0, 1)            # (max_len, 1, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """x: (seq_len, batch, d_model)"""
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

In [ ]:
# ── Visualise positional encoding ────────────────────────────────────────────
pe_module = PositionalEncoding(d_model=128, dropout=0.0)
pe_values = pe_module.pe[:100, 0, :].numpy()  # first 100 positions

plt.figure(figsize=(12, 4))
plt.imshow(pe_values, aspect='auto', cmap='RdBu', interpolation='nearest')
plt.colorbar(label='Value')
plt.xlabel('Embedding Dimension')
plt.ylabel('Position')
plt.title('Positional Encoding Heatmap (d_model=128, first 100 positions)')
plt.tight_layout()
plt.show()

> **Tutorial note — post-tutorial reading.** The encoder block is a *plumbing* layer on top of self-attention (residual + LayerNorm + FFN). In the live tutorial we discuss it verbally; the code below is for you to study and run after class.

### 6.5 Transformer Encoder Block

A single Transformer encoder block consists of:

1. **Multi-Head Self-Attention** -> 2. **Add & LayerNorm** -> 3. **Feed-Forward Network (FFN)** -> 4. **Add & LayerNorm**

Key design choices:

- **Residual connections** (Add): Same principle as ResNet (Week 5) -- allow gradients to flow directly through the network, enabling deeper stacking.
- **Layer Normalisation** (not Batch Normalisation): Normalises across **features** for each sample independently, making it more stable for **variable-length** sequences where batch statistics would be unreliable.
- **Feed-Forward Network**: Expands then contracts -- $d_{model} \to d_{ff} \to d_{model}$, typically with $d_{ff} = 4 \times d_{model}$. This provides the model with additional capacity for non-linear transformation at each position independently.

In [ ]:
# ── Transformer Encoder Block ────────────────────────────────────────────────

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """x: (B, T, d_model). Returns: (B, T, d_model), attn_weights."""
        # ── Self-attention + residual + norm ──
        attn_out, attn_weights = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        # ── FFN + residual + norm ──
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x, attn_weights

In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────
torch.manual_seed(42)
block = TransformerEncoderBlock(d_model=64, num_heads=4, d_ff=256)
x = torch.randn(2, 10, 64)  # batch=2, seq_len=10, d_model=64
out, attn = block(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention: {attn.shape}")

> **Tutorial note — post-tutorial reading.** Masking is introduced here because you will need it for Week 8 (GPT). In the live tutorial, you already implemented the causal mask in §6.2 — this section just formalises it.

### 6.6 Masking

For autoregressive tasks (e.g., language modelling where we predict the next token), we must prevent the model from "seeing the future". We apply a **causal mask** -- a lower-triangular matrix where future positions are set to $-\infty$ before softmax.

$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } j \leq i \quad (\text{allowed}) \\\\ -\infty & \text{if } j > i \quad (\text{blocked}) \end{cases}$$

After softmax, the $-\infty$ entries become 0, so each token can only attend to itself and earlier tokens.

In [ ]:
# ── Causal mask generation ───────────────────────────────────────────────────

def generate_square_subsequent_mask(sz):
    """Generate a causal (lower-triangular) mask for autoregressive attention.
    Returns: (sz, sz) float tensor with 0 for allowed and -inf for blocked positions.
    """
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

# Visualise a 6x6 causal mask
mask = generate_square_subsequent_mask(6)
print("Causal mask (0 = allowed, -inf = blocked):")
print(mask)

### 6.7 Language Modelling with nn.TransformerEncoder ⭐ *Extension*

> *This section demonstrates a production Transformer on Penn Treebank. In class, focus on the architecture discussion. Run the training as a take-home exercise.*

Now we put everything together: train a Transformer on a **language modelling** task using the Penn Treebank dataset. The model predicts the next word given all previous words.

We use PyTorch's built-in `nn.TransformerEncoder` here (which internally uses the same components we built from scratch above).

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install torchtext torchdata portalocker>=2.0.0

In [ ]:
# ── Transformer Language Model ───────────────────────────────────────────────

from torch.nn import TransformerEncoder, TransformerEncoderLayer

class TransformerLM(nn.Module):
    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):
        super(TransformerLM, self).__init__()
        self.model_type = 'Transformer'
        self.pos_encoder = PositionalEncoding(ninp, dropout)
        encoder_layers = TransformerEncoderLayer(ninp, nhead, nhid, dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)
        self.encoder = nn.Embedding(ntoken, ninp)
        self.ninp = ninp
        self.decoder = nn.Linear(ninp, ntoken)
        self.init_weights()

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def init_weights(self):
        initrange = 0.1
        self.encoder.weight.data.uniform_(-initrange, initrange)
        self.decoder.bias.data.zero_()
        self.decoder.weight.data.uniform_(-initrange, initrange)

    def forward(self, src, src_mask):
        src = self.encoder(src) * math.sqrt(self.ninp)  # scale embeddings
        src = self.pos_encoder(src)                      # add positional encoding
        output = self.transformer_encoder(src, src_mask)  # transformer encoder
        output = self.decoder(output)                     # project to vocabulary
        return output

In [ ]:
# ── Data loading: Penn Treebank ──────────────────────────────────────────────

from torchtext.datasets import PennTreebank
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

train_iter = PennTreebank(split='train')
tokenizer = get_tokenizer('basic_english')

vocab = build_vocab_from_iterator(map(tokenizer, train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

def data_process(raw_text_iter):
    """Convert raw text to a flat tensor of token IDs."""
    data = [torch.tensor(vocab(tokenizer(item)), dtype=torch.long)
            for item in raw_text_iter]
    return torch.cat(tuple(filter(lambda t: t.numel() > 0, data)))

train_iter, val_iter, test_iter = PennTreebank()
train_data_lm = data_process(train_iter)
val_data_lm = data_process(val_iter)
test_data_lm = data_process(test_iter)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batchify(data, bsz):
    """Arrange data into bsz columns for language modelling."""
    nbatch = data.size(0) // bsz
    data = data.narrow(0, 0, nbatch * bsz)
    data = data.view(bsz, -1).t().contiguous()
    return data.to(device)

lm_batch_size = 20
eval_batch_size = 10
train_data_lm = batchify(train_data_lm, lm_batch_size)
val_data_lm = batchify(val_data_lm, eval_batch_size)
test_data_lm = batchify(test_data_lm, eval_batch_size)

print(f"Vocabulary size: {len(vocab)}")
print(f"Train data shape: {train_data_lm.shape}")

In [ ]:
# ── Batching for language modelling ──────────────────────────────────────────

bptt = 35  # backpropagation through time window

def get_batch(source, i):
    """Get a batch of (input, target) pairs for language modelling.
    Input:  source[i : i+seq_len]
    Target: source[i+1 : i+1+seq_len]  (shifted by 1)
    """
    seq_len = min(bptt, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].reshape(-1)
    return data, target

In [ ]:
# ── Model instantiation ──────────────────────────────────────────────────────

ntokens = len(vocab)  # vocabulary size
emsize = 200          # embedding dimension
nhid = 200            # feedforward network dimension
nlayers = 2           # number of encoder layers
nhead = 2             # number of attention heads
dropout = 0.2         # dropout rate

transformer_model = TransformerLM(ntokens, emsize, nhead, nhid, nlayers, dropout).to(device)
n_params = sum(p.numel() for p in transformer_model.parameters())
print(f"Transformer parameters: {n_params:,}")

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss()
lr = 5.0
optimizer_lm = torch.optim.SGD(transformer_model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer_lm, 1.0, gamma=0.95)


def train_lm(epoch):
    """Train for one epoch."""
    transformer_model.train()
    total_loss = 0.
    start_time = time.time()
    src_mask = transformer_model.generate_square_subsequent_mask(bptt).to(device)

    for batch_idx, i in enumerate(range(0, train_data_lm.size(0) - 1, bptt)):
        data, targets = get_batch(train_data_lm, i)
        optimizer_lm.zero_grad()
        if data.size(0) != bptt:
            src_mask = transformer_model.generate_square_subsequent_mask(data.size(0)).to(device)
        output = transformer_model(data, src_mask)
        loss = criterion(output.view(-1, ntokens), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), 0.5)
        optimizer_lm.step()

        total_loss += loss.item()
        log_interval = 200
        if batch_idx % log_interval == 0 and batch_idx > 0:
            cur_loss = total_loss / log_interval
            elapsed = time.time() - start_time
            print(f'| epoch {epoch:3d} | {batch_idx:5d}/{len(train_data_lm) // bptt:5d} batches | '
                  f'lr {scheduler.get_last_lr()[0]:.2f} | ms/batch {elapsed * 1000 / log_interval:.2f} | '
                  f'loss {cur_loss:.2f} | ppl {math.exp(cur_loss):.2f}')
            total_loss = 0
            start_time = time.time()


def evaluate_lm(eval_model, data_source):
    """Evaluate perplexity on a dataset."""
    eval_model.eval()
    total_loss = 0.
    src_mask = transformer_model.generate_square_subsequent_mask(bptt).to(device)
    with torch.no_grad():
        for i in range(0, data_source.size(0) - 1, bptt):
            data, targets = get_batch(data_source, i)
            if data.size(0) != bptt:
                src_mask = transformer_model.generate_square_subsequent_mask(data.size(0)).to(device)
            output = eval_model(data, src_mask)
            output_flat = output.view(-1, ntokens)
            total_loss += len(data) * criterion(output_flat, targets).item()
    return total_loss / (len(data_source) - 1)

In [ ]:
# ── Train for 3 epochs ───────────────────────────────────────────────────────

best_val_loss = float("inf")
epochs = 3
best_model = None

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    train_lm(epoch)
    val_loss = evaluate_lm(transformer_model, val_data_lm)
    print('-' * 89)
    print(f'| end of epoch {epoch:3d} | time: {time.time() - epoch_start_time:.2f}s | '
          f'valid loss {val_loss:.2f} | valid ppl {math.exp(val_loss):.2f}')
    print('-' * 89)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = transformer_model

    scheduler.step()

In [ ]:
# ── Test evaluation ──────────────────────────────────────────────────────────

test_loss = evaluate_lm(best_model, test_data_lm)
print('=' * 89)
print(f'| End of training | test loss {test_loss:.2f} | test ppl {math.exp(test_loss):.2f}')
print('=' * 89)

---

## 7. Grand Comparison

### 7.1 RNN vs LSTM vs GRU: Performance on Letter-Counting Task

See the three-way comparison plot above (Section 4.3). Key observations:
- LSTM and GRU significantly outperform vanilla RNN due to gated memory.
- GRU converges slightly faster (fewer parameters) but reaches similar final accuracy as LSTM.
- Vanilla RNN struggles to learn long-range counting dependencies.

### 7.2 Architectural Comparison

| Feature | RNN | LSTM | GRU | Transformer |
|---|---|---|---|---|
| **Memory mechanism** | Hidden state | Cell state + hidden state | Gated hidden state | Attention weights |
| **Gates** | 0 | 3 (forget, input, output) | 2 (reset, update) | 0 (learned projections) |
| **Sequential processing** | Yes | Yes | Yes | **No** |
| **Long-range dependencies** | Poor | Good | Good | **Excellent** |
| **Vanishing gradient** | Severe | Mitigated (additive cell) | Mitigated (additive state) | **None** (direct attention) |
| **Bidirectional** | Requires 2x cost | Requires 2x cost | Requires 2x cost | **Native** |
| **Computational complexity** | $O(T \cdot d^2)$ | $O(T \cdot d^2)$ | $O(T \cdot d^2)$ | $O(T^2 \cdot d)$ |

### 7.3 When to Use What

- **RNN**: Rarely used in practice. Important for understanding the fundamentals.
- **LSTM**: When you need explicit forget/remember control, or when the task requires modelling very long sequences with precise gating.
- **GRU**: Good default for recurrent tasks. Fewer parameters than LSTM, often similar performance.
- **Transformer**: Default choice for most sequence tasks when computational resources allow. Scales well with parallelism, excels at long-range dependencies. Note: $O(T^2)$ attention cost can be prohibitive for very long sequences ($T > 10{,}000$).

---
# Part C · Exam-Style Questions

Three medium-to-hard short-answer questions covering the Transformer content from this tutorial. Each question deliberately reaches back to earlier weeks so you practise *connecting* ideas rather than reciting isolated facts.

Try each question on paper first. Expected-answer sketches follow each question in a collapsed cell — open them only after you have attempted the question.

### Q1 — Why $\sqrt{d_k}$? (Week 3 connection: softmax / gradient flow)

Scaled dot-product attention computes

$$
\text{Attn}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V.
$$

Suppose the entries of $Q$ and $K$ are i.i.d. with zero mean and unit variance.

**(a)** Show that each entry of $Q K^\top$ has variance $d_k$.

**(b)** Explain what happens to the output of the softmax as $d_k \to \infty$ **without** the $\sqrt{d_k}$ scaling.

**(c)** Using what you know about the softmax Jacobian, explain why this is a problem for gradient-based training, and argue why dividing by $\sqrt{d_k}$ — as opposed to $d_k$ or $1$ — is the right fix.


<details><summary><strong>Answer sketch — Q1</strong> (open after attempting)</summary>

**(a)** Let $q, k \in \mathbb{R}^{d_k}$ be a query/key pair. Their inner product is $\langle q, k \rangle = \sum_{i=1}^{d_k} q_i k_i$. Each $q_i k_i$ has mean $\mathbb{E}[q_i]\mathbb{E}[k_i] = 0$ and variance $\mathrm{Var}(q_i)\mathrm{Var}(k_i) = 1$, and the terms are independent. Therefore $\mathrm{Var}(\langle q, k \rangle) = d_k$ and the standard deviation grows like $\sqrt{d_k}$.

**(b)** Without scaling, the logits fed into softmax have magnitude on the order of $\sqrt{d_k}$. For large $d_k$, the largest logit dominates and the softmax output concentrates on a single position — it approaches a one-hot distribution.

**(c)** The Jacobian of softmax is $J_{ij} = p_i(\delta_{ij} - p_j)$. When the output is near one-hot ($p \approx e_{i^\star}$), every entry of $J$ is near zero, so **gradients vanish into $Q$ and $K$**. The fix needs to keep the *variance* of the logits at $O(1)$, which is exactly what dividing by $\sqrt{d_k}$ achieves. Dividing by $d_k$ instead would shrink the standard deviation to $1/\sqrt{d_k} \to 0$, over-flattening the softmax and destroying the model's ability to *select* anything — the opposite failure mode.

</details>


### Q2 — RNN vs. self-attention: FLOPs, path length, parallelism (Week 5 connection: complexity analysis)

Consider processing a sequence of length $T$ with hidden/model dimension $d$.

**(a)** Give the time complexity of one forward pass through (i) a single vanilla RNN layer and (ii) a single self-attention layer.

**(b)** Give the **maximum path length** between any two tokens in each model — i.e., the minimum number of layer operations a signal must traverse to connect token $i$ to token $j$.

**(c)** Training throughput matters as much as asymptotic cost. Which of the two architectures can be parallelised across the time dimension during training, and why does the other one fundamentally cannot?

**(d)** For what regime of $T$ and $d$ is self-attention cheaper than an RNN in raw FLOPs, ignoring parallelism?


<details><summary><strong>Answer sketch — Q2</strong> (open after attempting)</summary>

**(a)** RNN per step: one matmul of size $O(d^2)$, repeated for $T$ steps → $O(T d^2)$. Self-attention: the $Q K^\top$ product and the attention-weighted sum of $V$ cost $O(T^2 d)$; the $Q/K/V/O$ projections cost $O(T d^2)$. Total: $O(T^2 d + T d^2)$.

**(b)** RNN: $O(T)$ — information from token 1 reaches token $T$ only by traversing $T-1$ recurrent steps. Self-attention: $O(1)$ — every token attends directly to every other token in a single layer.

**(c)** Self-attention can be fully parallelised across the time axis during training because the computation of the representation at position $t$ does **not** depend on the representation at position $t-1$; all $T$ positions are computed from the same input tensor. The RNN hidden state satisfies $h_t = f(h_{t-1}, x_t)$, so computing $h_t$ requires $h_{t-1}$ to already be known — the recurrence is inherently sequential and cannot be unrolled on a GPU.

**(d)** Comparing the dominant terms, self-attention is cheaper in FLOPs when $T^2 d < T d^2$, i.e. when $T < d$. In practice modern Transformers often run with $T \sim d$ or $T > d$, so Transformers win on *path length and parallelism*, not on asymptotic FLOPs — which is the important takeaway.

</details>


### Q3 — Permutation equivariance and positional encoding (Week 5/6 connection: symmetry and inductive biases)

Let $\text{Attn}(X)$ denote self-attention applied to a sequence $X \in \mathbb{R}^{T \times d}$, with $Q = X W_Q$, $K = X W_K$, $V = X W_V$. Let $P \in \mathbb{R}^{T \times T}$ be any permutation matrix.

**(a)** Prove that $\text{Attn}(P X) = P \cdot \text{Attn}(X)$. (I.e., self-attention is **permutation-equivariant** over tokens.)

**(b)** A classmate argues:
> *"Since self-attention is permutation-equivariant, a Transformer cannot distinguish 'dog bites man' from 'man bites dog'. Adding positional encodings to the input embeddings fixes this because the positional vectors break the symmetry."*

Is the classmate's **conclusion** correct? Is the **reasoning** rigorous? If not, what is the subtle gap?

**(c)** In Week 5 (CNN) we relied on translation equivariance, and in Week 6 (GNN) we relied on permutation equivariance over nodes, as *desirable* inductive biases. For sequence modelling, is permutation equivariance a desirable inductive bias or an obstacle? Justify in one or two sentences.


<details><summary><strong>Answer sketch — Q3</strong> (open after attempting)</summary>

**(a)** With $X' = P X$ we get $Q' = P X W_Q = P Q$ and likewise $K' = P K$, $V' = P V$. Then
$$Q' K'^{\top} = P Q K^\top P^\top.$$
Softmax is applied **row-wise**, and row permutation commutes with any row-wise function, so
$$\mathrm{softmax}\!\left(\tfrac{Q' K'^\top}{\sqrt{d_k}}\right) = P\, \mathrm{softmax}\!\left(\tfrac{Q K^\top}{\sqrt{d_k}}\right) P^\top.$$
Multiplying by $V' = P V$ and using $P^\top P = I$:
$$\text{Attn}(P X) = P\, \mathrm{softmax}(\cdots)\, P^\top P V = P\, \mathrm{softmax}(\cdots) V = P\, \text{Attn}(X). \quad\blacksquare$$

**(b) Conclusion correct, reasoning loose.** The conclusion that a vanilla Transformer cannot distinguish the two sentences is right. But "adding positional encodings breaks the symmetry" is not automatic. The symmetry is only broken because positional encodings are a **fixed reference frame**: $\text{PE}$ depends on position index, not on token identity, and is **not** permuted along with the tokens. If you permuted $\text{PE}$ together with $X$ (i.e., treated $\text{PE}$ as part of the token features), you would end up with $\text{Attn}(P(X + \text{PE})) = P\,\text{Attn}(X + \text{PE})$ by part (a), and the problem would reappear. The gap in the classmate's argument is that they did not explain *why* $\text{PE}$ is not permuted — that is the actual symmetry-breaking step.

**(c)** It is an **obstacle**. In sequence data (language, time series) the *order* is meaningful — "dog bites man" ≠ "man bites dog" — so a good sequence model must be position-*sensitive*. This is the opposite of GNNs, where node labels are arbitrary and permutation equivariance is exactly what we want, and different from CNNs, where translation equivariance holds because pixel coordinates lie in a metric space with meaningful shifts. Positional encoding exists precisely to cancel the symmetry that self-attention would otherwise enforce.

</details>
